<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c11/c11_ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11 - ex 8
Practice training a DNN on the CIFAR10 image dataset

## a. Load CIFAR10

If it is the first time downloading the dataset

In [1]:
import torch
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.CIFAR10(
    root='datasets', train=True, download=True, transform=toTensor
)
test_data = torchvision.datasets.CIFAR10(
    root='datasets', train=False, download=True, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [45_000, 5_000]
)

100%|██████████| 170M/170M [26:58<00:00, 105kB/s]


Save the dataset to google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/datasets /content/drive/MyDrive/ #copy the whole datasets folder

Mounted at /content/drive


If the dataset was already downloaded and stored at /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.CIFAR10(
    root='/content/drive/MyDrive/datasets', train=True, download=False, transform=toTensor
)
test_data = torchvision.datasets.CIFAR10(
    root='/content/drive/MyDrive/datasets', train=False, download=False, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [45_000, 5_000]
)

In [3]:
# create data loaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [4]:
# classes in the data set
print(train_and_valid_data.classes)
images, labels = next(iter(train_loader))
print(labels.min(), labels.max())
print(images.shape)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
tensor(0) tensor(9)
torch.Size([32, 3, 32, 32])


In [5]:
import torch.nn as nn

In [6]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

## b. Build a DNN with 20 hidden layers of 100 neurons each
Use He initialization and the Swish activation function. Classification task: it requires an output layer with one neuron per class

In [7]:
class ImageClassifier(nn.Module):
  def __init__(self, n_inputs, n_neurons, n_layers, n_classes):
    super().__init__()
    # layers is a temporary list used to construct the model, no need to save it as self.layers
    layers = [nn.Flatten(),
                   nn.Linear(n_inputs,n_neurons),
                   nn.SiLU()]
    for i in range(n_layers-1):
      layers.append(nn.Linear(n_neurons,n_neurons))
      layers.append(nn.SiLU())
    layers.append(nn.Linear(n_neurons, n_classes))

    # initialize weights using He
    def use_he_init(module):
      if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

    self.model = nn.Sequential(*layers)
    self.model.apply(use_he_init)

  def forward(self,X):
    return self.model(X)

## c. Using NAdam optimization and early stopping, train the network
Remember to search the right learning rate each time you change the model's architecture or hyperparameters.

### Define the train function

In [8]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 22.9 MB/s eta 0:00:00


In [9]:
import torchmetrics

In [44]:
import copy
import time

In [45]:
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs, patience=10):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
  valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to(device)

  best_acc = 0.0
  best_state = None
  patience_counter = 0

  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()
    total_loss = 0.
    t0 = time.time()

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      loss = criterion(y_pred, y_train_batch)
      total_loss += loss.item()
      train_accuracy.update(y_pred, y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    mean_loss = total_loss/len(train_loader)

    with torch.no_grad():
      model.eval()
      valid_accuracy.reset()

      for X_valid_batch, y_valid_batch in valid_loader:
        X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
        y_valid_pred = model(X_valid_batch)
        valid_accuracy.update(y_valid_pred, y_valid_batch)
      epoch_valid_accuracy = valid_accuracy.compute()

      # we save the best model. We don't stop if validation decresaes bc w/ mini-batch sometimes accuracy decreases but then increases again.
      if epoch_valid_accuracy > best_acc:
        best_acc = epoch_valid_accuracy
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
      else:
        patience_counter +=1

    if patience_counter == patience:
      print(f'Early stopping')
      break
    t1 = time.time()

    print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}, Validation Accuracy: {epoch_valid_accuracy:.4f}'
          f'in {t1-t0:.1f}s')
  if best_state is not None:
    model.load_state_dict(best_state)

  return model, best_acc



### Initialize the model

In [29]:
# The classes are 0, ..., 9
# The images have shape [3,32,32]

torch.manual_seed(42)
model1 = ImageClassifier(n_inputs = 3 * 32 * 32, n_neurons = 100, n_layers = 20, n_classes = 10)
model1 = model1.to(device)

In [30]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model1.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [31]:
best_model, best_acc = train(model1, optimizer, xentropy, train_loader, valid_loader, 50, 5)

Epoch 1/50: Train Loss: 2.0066 Train Accuracy: 0.2405, Validation Accuracy: 0.2482
Epoch 2/50: Train Loss: 1.8296 Train Accuracy: 0.3218, Validation Accuracy: 0.3474
Epoch 3/50: Train Loss: 1.7483 Train Accuracy: 0.3605, Validation Accuracy: 0.3320
Epoch 4/50: Train Loss: 1.6892 Train Accuracy: 0.3894, Validation Accuracy: 0.3614
Epoch 5/50: Train Loss: 1.6478 Train Accuracy: 0.4042, Validation Accuracy: 0.3862
Epoch 6/50: Train Loss: 1.6172 Train Accuracy: 0.4206, Validation Accuracy: 0.4068
Epoch 7/50: Train Loss: 1.5920 Train Accuracy: 0.4326, Validation Accuracy: 0.4074
Epoch 8/50: Train Loss: 1.5696 Train Accuracy: 0.4383, Validation Accuracy: 0.4110
Epoch 9/50: Train Loss: 1.5502 Train Accuracy: 0.4457, Validation Accuracy: 0.4326
Epoch 10/50: Train Loss: 1.5372 Train Accuracy: 0.4538, Validation Accuracy: 0.4344
Epoch 11/50: Train Loss: 1.5178 Train Accuracy: 0.4634, Validation Accuracy: 0.4348
Epoch 12/50: Train Loss: 1.5029 Train Accuracy: 0.4649, Validation Accuracy: 0.4340
E

In [33]:
best_acc

tensor(0.4594, device='cuda:0')

## d. Add batch-norm and compare the learning curves

In [35]:
class ImageClassifier2(nn.Module):
  def __init__(self, n_inputs, n_neurons, n_layers, n_classes):
    super().__init__()
    # layers is a temporary list used to construct the model, no need to save it as self.layers
    layers = [nn.Flatten(),
                   nn.BatchNorm1d(n_inputs),
                   nn.Linear(n_inputs,n_neurons),
                   nn.SiLU()]
    for i in range(n_layers-1):
      layers.append(nn.BatchNorm1d(n_neurons))
      layers.append(nn.Linear(n_neurons,n_neurons))
      layers.append(nn.SiLU())
    layers.append(nn.BatchNorm1d(n_neurons))
    layers.append(nn.Linear(n_neurons, n_classes))

    # initialize weights using He
    def use_he_init(module):
      if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

    self.model = nn.Sequential(*layers)
    self.model.apply(use_he_init)

  def forward(self,X):
    return self.model(X)

In [36]:
torch.manual_seed(42)
model2 = ImageClassifier2(n_inputs = 3 * 32 * 32, n_neurons = 100, n_layers = 20, n_classes = 10)
model2 = model2.to(device)

In [37]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model2.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [38]:
best_model2, best_acc2 = train(model2, optimizer, xentropy, train_loader, valid_loader, 50, 5)

Epoch 1/50: Train Loss: 2.1502 Train Accuracy: 0.2209, Validation Accuracy: 0.3106
Epoch 2/50: Train Loss: 1.8928 Train Accuracy: 0.3134, Validation Accuracy: 0.3596
Epoch 3/50: Train Loss: 1.8023 Train Accuracy: 0.3514, Validation Accuracy: 0.3940
Epoch 4/50: Train Loss: 1.7404 Train Accuracy: 0.3774, Validation Accuracy: 0.3974
Epoch 5/50: Train Loss: 1.6922 Train Accuracy: 0.4019, Validation Accuracy: 0.4154
Epoch 6/50: Train Loss: 1.6467 Train Accuracy: 0.4162, Validation Accuracy: 0.4278
Epoch 7/50: Train Loss: 1.6088 Train Accuracy: 0.4320, Validation Accuracy: 0.4628
Epoch 8/50: Train Loss: 1.5665 Train Accuracy: 0.4479, Validation Accuracy: 0.4578
Epoch 9/50: Train Loss: 1.5322 Train Accuracy: 0.4615, Validation Accuracy: 0.4658
Epoch 10/50: Train Loss: 1.4999 Train Accuracy: 0.4720, Validation Accuracy: 0.4782
Epoch 11/50: Train Loss: 1.4687 Train Accuracy: 0.4842, Validation Accuracy: 0.4768
Epoch 12/50: Train Loss: 1.4349 Train Accuracy: 0.4950, Validation Accuracy: 0.4944
E

## e. Replace batch-norm with SELU
Make the necessary adjustments to ensure the network self-normalizes
(i.e. standardize input features, use LeCun normal initialization, make sure DNN contains only a sequence of dense layers, etc)

In [39]:
class ImageClassifier_SELU(nn.Module):
  def __init__(self, n_inputs, n_neurons, n_layers, n_classes):
    super().__init__()
    # layers is a temporary list used to construct the model, no need to save it as self.layers
    layers = [nn.Flatten(),
                   nn.BatchNorm1d(n_inputs),
                   nn.Linear(n_inputs,n_neurons),
                   nn.SELU()]
    for i in range(n_layers-1):
      layers.append(nn.Linear(n_neurons,n_neurons))
      layers.append(nn.SELU())
    layers.append(nn.Linear(n_neurons, n_classes))

    # initialize weights using He
    def use_lecun_init(module):
      if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight,mode = 'fan_in', nonlinearity='linear')
        nn.init.zeros_(module.bias)

    self.model = nn.Sequential(*layers)
    self.model.apply(use_lecun_init)

  def forward(self,X):
    return self.model(X)

In [40]:
torch.manual_seed(42)
model3 = ImageClassifier_SELU(n_inputs = 3 * 32 * 32, n_neurons = 100, n_layers = 20, n_classes = 10)
model3 = model3.to(device)

In [41]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model3.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [46]:
best_model3, best_acc3 = train(model3, optimizer, xentropy, train_loader, valid_loader, 50, 5)

Epoch 1/50: Train Loss: 1.9876 Train Accuracy: 0.2785, Validation Accuracy: 0.3370in 20.1s
Epoch 2/50: Train Loss: 1.7916 Train Accuracy: 0.3572, Validation Accuracy: 0.3470in 21.3s
Epoch 3/50: Train Loss: 1.7117 Train Accuracy: 0.3897, Validation Accuracy: 0.3936in 21.0s
Epoch 4/50: Train Loss: 1.6570 Train Accuracy: 0.4123, Validation Accuracy: 0.4104in 21.1s
Epoch 5/50: Train Loss: 1.6131 Train Accuracy: 0.4320, Validation Accuracy: 0.4394in 20.6s
Epoch 6/50: Train Loss: 1.5789 Train Accuracy: 0.4364, Validation Accuracy: 0.4322in 21.1s
Epoch 7/50: Train Loss: 1.5532 Train Accuracy: 0.4542, Validation Accuracy: 0.4324in 20.7s
Epoch 8/50: Train Loss: 1.5674 Train Accuracy: 0.4468, Validation Accuracy: 0.4526in 20.9s
Epoch 9/50: Train Loss: 1.5060 Train Accuracy: 0.4704, Validation Accuracy: 0.4670in 20.8s
Epoch 10/50: Train Loss: 1.4691 Train Accuracy: 0.4790, Validation Accuracy: 0.4332in 21.0s
Epoch 11/50: Train Loss: 1.4627 Train Accuracy: 0.4886, Validation Accuracy: 0.4574in 20.

## f. Regularize the model with alpha droput.

This is dropout when using a self-normalizing network based on the SELU activation function.

In [47]:
class ImageClassifier_a_dropout(nn.Module):
  def __init__(self, n_inputs, n_neurons, n_layers, n_classes, dropout_rate):
    super().__init__()
    # layers is a temporary list used to construct the model, no need to save it as self.layers
    layers = [nn.Flatten(),
                   nn.BatchNorm1d(n_inputs),
                   nn.AlphaDropout(p= dropout_rate),
                   nn.Linear(n_inputs,n_neurons),
                   nn.SELU()]
    for i in range(n_layers-1):
      layers.append(nn.AlphaDropout(p= dropout_rate))
      layers.append(nn.Linear(n_neurons,n_neurons))
      layers.append(nn.SELU())
    layers.append(nn.AlphaDropout(p= dropout_rate))
    layers.append(nn.Linear(n_neurons, n_classes))

    # initialize weights using He
    def use_lecun_init(module):
      if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight,mode = 'fan_in', nonlinearity='linear')
        nn.init.zeros_(module.bias)

    self.model = nn.Sequential(*layers)
    self.model.apply(use_lecun_init)

  def forward(self,X):
    return self.model(X)

In [48]:
torch.manual_seed(42)
model4 = ImageClassifier_a_dropout(n_inputs = 3 * 32 * 32, n_neurons = 100, n_layers = 20, n_classes = 10, dropout_rate = 0.2)
model4 = model4.to(device)

In [49]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model4.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [50]:
best_model4, best_acc4 = train(model4, optimizer, xentropy, train_loader, valid_loader, 50, 5)

Epoch 1/50: Train Loss: 2.2741 Train Accuracy: 0.1429, Validation Accuracy: 0.1506in 24.9s
Epoch 2/50: Train Loss: 2.1278 Train Accuracy: 0.1678, Validation Accuracy: 0.1566in 25.5s
Epoch 3/50: Train Loss: 2.0726 Train Accuracy: 0.1966, Validation Accuracy: 0.1696in 25.9s
Epoch 4/50: Train Loss: 2.0529 Train Accuracy: 0.2086, Validation Accuracy: 0.1662in 25.5s
Epoch 5/50: Train Loss: 2.0763 Train Accuracy: 0.2009, Validation Accuracy: 0.1572in 25.7s
Epoch 6/50: Train Loss: 2.0541 Train Accuracy: 0.2119, Validation Accuracy: 0.1938in 25.8s
Epoch 7/50: Train Loss: 2.0282 Train Accuracy: 0.2174, Validation Accuracy: 0.1804in 24.8s
Epoch 8/50: Train Loss: 2.0232 Train Accuracy: 0.2226, Validation Accuracy: 0.1970in 24.8s
Epoch 9/50: Train Loss: 2.0152 Train Accuracy: 0.2218, Validation Accuracy: 0.2020in 25.5s
Epoch 10/50: Train Loss: 1.9979 Train Accuracy: 0.2286, Validation Accuracy: 0.1882in 25.3s
Epoch 11/50: Train Loss: 2.0041 Train Accuracy: 0.2265, Validation Accuracy: 0.2028in 25.

In [57]:
best_acc4

tensor(0.2270, device='cuda:0')

### Without retraining the model, use MC dropout to evaluate

In [56]:
best_model4.eval()
for module in best_model4.modules():
  if isinstance(module, nn.AlphaDropout):
    module.train()

torch.manual_seed(42)

valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to(device)

with torch.no_grad():
  for X_batch, y_batch in valid_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        X_batch_repeated = X_batch.repeat_interleave(100, dim=0)
        y_batch_logits_all = best_model4(X_batch_repeated).reshape(len(X_batch), 100, 10)
        y_batch_probas_all = torch.nn.functional.softmax(y_batch_logits_all, dim=-1)
        y_batch_probas = y_batch_probas_all.mean(dim=1)
        valid_accuracy.update(y_batch_probas, y_batch)
  final_valid_accuracy = valid_accuracy.compute()

print(f'validation accuracy: {final_valid_accuracy:.4f}')


validation accuracy: 0.2510


Try with a reduced dropout probability of 0.1

In [58]:
torch.manual_seed(42)
model5 = ImageClassifier_a_dropout(n_inputs = 3 * 32 * 32, n_neurons = 100, n_layers = 20, n_classes = 10, dropout_rate = 0.1)
model5 = model5.to(device)

In [59]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model5.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [60]:
best_model5, best_acc5 = train(model5, optimizer, xentropy, train_loader, valid_loader, 50, 5)

Epoch 1/50: Train Loss: 2.1343 Train Accuracy: 0.1973, Validation Accuracy: 0.2158in 25.9s
Epoch 2/50: Train Loss: 1.9777 Train Accuracy: 0.2451, Validation Accuracy: 0.2556in 26.7s
Epoch 3/50: Train Loss: 1.9329 Train Accuracy: 0.2667, Validation Accuracy: 0.2676in 25.6s
Epoch 4/50: Train Loss: 1.9138 Train Accuracy: 0.2728, Validation Accuracy: 0.2934in 25.4s
Epoch 5/50: Train Loss: 1.9101 Train Accuracy: 0.2766, Validation Accuracy: 0.2948in 24.9s
Epoch 6/50: Train Loss: 1.9007 Train Accuracy: 0.2807, Validation Accuracy: 0.2940in 24.6s
Epoch 7/50: Train Loss: 1.8898 Train Accuracy: 0.2898, Validation Accuracy: 0.3018in 25.5s
Epoch 8/50: Train Loss: 1.8962 Train Accuracy: 0.2896, Validation Accuracy: 0.2948in 25.2s
Epoch 9/50: Train Loss: 1.8920 Train Accuracy: 0.2869, Validation Accuracy: 0.2748in 25.5s
Epoch 10/50: Train Loss: 1.9187 Train Accuracy: 0.2708, Validation Accuracy: 0.2742in 25.6s
Epoch 11/50: Train Loss: 1.8919 Train Accuracy: 0.2864, Validation Accuracy: 0.2654in 25.